In [1]:
# uploading image path
MY_IMAGE = "nandini.jpeg"

In [2]:
# imporitng libraries
import os
import cv2
import joblib
import torch
import pyiqa
import shutil
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from insightface.app import FaceAnalysis

In [3]:
# loading arcface model
print("Loading ArcFace...")

app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(640, 640))

rec_model = app.models["recognition"]
print("ArcFace Loaded")

Loading ArcFace...


/opt/miniconda3/lib/python3.13/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:149: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'CoreMLExecutionProvider, AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/admin/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
se

In [4]:
# loading MUSIQ for quality testing
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

musiq_metric = pyiqa.create_metric("musiq", device=device)

print("MUSIQ Loaded")

Loading pretrained model MUSIQ from /Users/admin/.cache/torch/hub/pyiqa/musiq_koniq_ckpt-e95806b9.pth
MUSIQ Loaded


In [5]:
# loading saved svm model
svm = joblib.load("svm_rejection_model.pkl")

scaler = joblib.load("svm_scaler.pkl")

print("SVM Loaded")

SVM Loaded


In [6]:
# creating working folders
ROOT = "my_test"

os.makedirs(ROOT, exist_ok=True)
os.makedirs(os.path.join(ROOT, "degraded"), exist_ok=True)
os.makedirs(os.path.join(ROOT, "embeddings"), exist_ok=True)

In [7]:
# generating degraded images
img = cv2.imread(MY_IMAGE)

# blur low
blur_low = cv2.GaussianBlur(img, (5, 5), 0)
cv2.imwrite(os.path.join(ROOT, "degraded", "blur_low.jpg"), blur_low)

# blur high
blur_high = cv2.GaussianBlur(img, (15, 15), 0)
cv2.imwrite(os.path.join(ROOT, "degraded", "blur_high.jpg"), blur_high)

# compression low
cv2.imwrite(
    os.path.join(ROOT, "degraded", "jpeg_low.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 40]
)

# compression high
cv2.imwrite(
    os.path.join(ROOT, "degraded", "jpeg_high.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 10]
)

print("Degradations Created")

Degradations Created


In [8]:
# embedding function
def get_embedding(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    faces = app.get(img)

    if len(faces) == 0:
        return None

    emb = faces[0].embedding
    emb = emb / np.linalg.norm(emb)

    return emb

In [9]:
# generating embeddings
for file in os.listdir(os.path.join(ROOT, "degraded")):

    img_path = os.path.join(ROOT, "degraded", file)
    emb = get_embedding(img_path)

    if emb is None:
        continue

    np.save(os.path.join(ROOT, "embeddings", file.replace(".jpg", ".npy")), emb)

print("Embeddings Saved")

/opt/miniconda3/lib/python3.13/site-packages/insightface/utils/face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


Embeddings Saved


In [10]:
# laoding gallery
gallery_root = "embeddings/test/gallery"
gallery_db = {}

for identity in os.listdir(gallery_root):
    gallery_db[identity] = []
    identity_dir = os.path.join(gallery_root, identity)

    for file in os.listdir(identity_dir):
        emb = np.load(os.path.join(identity_dir, file))
        gallery_db[identity].append(emb)

print("Gallery Loaded")

Gallery Loaded


In [11]:
# utility functions
def quality_score(image_path):
    score = musiq_metric(image_path)
    return float(score.cpu().item())


def search_gallery(probe_embedding):
    scores = {}

    for identity in gallery_db:
        sims = []

        for g_emb in gallery_db[identity]:
            sim = cosine_similarity(
                probe_embedding.reshape(1, -1), g_emb.reshape(1, -1)
            )[0][0]
            sims.append(sim)

        scores[identity] = max(sims)

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    predicted_identity = sorted_scores[0][0]
    best_similarity = sorted_scores[0][1]
    second_similarity = sorted_scores[1][1]
    margin = best_similarity - second_similarity

    return (predicted_identity, best_similarity, second_similarity, margin)

In [12]:
# running full pipeline
results = []

for emb_file in os.listdir(os.path.join(ROOT, "embeddings")):
    emb_path = os.path.join(ROOT, "embeddings", emb_file)
    image_path = os.path.join(ROOT, "degraded", emb_file.replace(".npy", ".jpg"))
    probe_embedding = np.load(emb_path)
    q = quality_score(image_path)

    predicted_identity, best_similarity, second_similarity, margin = search_gallery(
        probe_embedding
    )

    features = pd.DataFrame(
        [[q, best_similarity, margin]],
        columns=["quality_score", "best_similarity", "margin"],
    )

    features_scaled = scaler.transform(features)
    decision = svm.predict(features_scaled)[0]

    results.append(
        {
            "image": emb_file,
            "quality": q,
            "predicted_identity": predicted_identity,
            "best_similarity": best_similarity,
            "second_similarity": second_similarity,
            "margin": margin,
            "decision": "ACCEPT" if decision == 1 else "REJECT",
        }
    )

In [13]:
# results
results_df = pd.DataFrame(results)
results_df

,image,quality,predicted_identity,best_similarity,second_similarity,margin,decision
0,jpeg_high.npy,46.395733,Amelie_Mauresmo,0.184578,0.145624,0.038954,ACCEPT
1,blur_low.npy,60.832985,Nancy_Pelosi,0.177096,0.131094,0.046002,ACCEPT
2,blur_high.npy,27.555948,Nancy_Pelosi,0.169775,0.134290,0.035484,ACCEPT
3,jpeg_low.npy,69.627937,Nancy_Pelosi,0.170092,0.135184,0.034908,ACCEPT
